# AWQ 量化教程

本教程介绍 Activation-aware Weight Quantization (AWQ)，一种高效的 LLM 量化方法。

## 目录
1. 量化基础
2. AWQ 原理
3. 实现细节
4. 实战演示

## 1. 量化基础

量化将 FP16/FP32 权重转换为低精度 (INT4/INT8)：

$$W_q = \text{round}\left(\frac{W}{s}\right) + z$$

其中 $s$ 是缩放因子，$z$ 是零点。

**优势**：
- 内存减少 4x (FP16 → INT4)
- 推理加速 2-4x

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
from deployment_optimization.model_optimization.src.awq import (
    AWQConfig,
    AWQQuantizer,
    AWQLinear,
    create_awq_quantizer
)

# 计算模型大小
def model_size_gb(params_billion, bits):
    return params_billion * bits / 8

print("LLaMA-7B 模型大小:")
print(f"  FP16: {model_size_gb(7, 16):.1f} GB")
print(f"  INT8: {model_size_gb(7, 8):.1f} GB")
print(f"  INT4: {model_size_gb(7, 4):.1f} GB")

## 2. AWQ 原理

AWQ 的核心观察：不是所有权重同等重要。

**关键思想**：
1. 通过激活值识别重要权重通道
2. 对重要通道使用更大的缩放因子
3. 保护重要权重，减少量化误差

$$s_i = \left(\frac{\max(|X_i|)}{\max(|X|)}\right)^\alpha$$

In [ ]:
# 创建 AWQ 配置
config = AWQConfig(
    w_bit=4,           # 4-bit 量化
    group_size=128,    # 分组大小
    zero_point=True    # 使用零点
)

print(f"量化位数: {config.w_bit}")
print(f"分组大小: {config.group_size}")

## 3. 实现细节

AWQ 量化流程：
1. 收集校准数据的激活值
2. 计算每个通道的重要性
3. 搜索最优缩放因子
4. 应用量化

In [ ]:
# 创建量化器
quantizer = create_awq_quantizer(
    w_bit=4,
    group_size=128
)

# 模拟权重
weight = np.random.randn(512, 256).astype(np.float32)
print(f"原始权重形状: {weight.shape}")
print(f"原始大小: {weight.nbytes / 1024:.1f} KB")

In [ ]:
# 量化权重
awq_linear = AWQLinear(
    in_features=256,
    out_features=512,
    config=config
)

# 执行量化
awq_linear.quantize_weight(weight)
print("量化完成!")

## 4. 实战演示

In [ ]:
# 计算量化误差
error_info = quantizer.compute_quantization_error(weight)

print(f"量化误差统计:")
for key, value in error_info.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.6f}")

In [ ]:
# 估算模型大小
size_info = quantizer.estimate_model_size(
    num_params=7_000_000_000,  # 7B 参数
    w_bit=4
)

print(f"模型大小估算:")
for key, value in size_info.items():
    print(f"  {key}: {value}")

## 总结

AWQ 的核心优势：

1. **高压缩比**: 4-bit 量化，4x 压缩
2. **低精度损失**: 激活感知保护重要权重
3. **无需重训练**: 仅需少量校准数据
4. **硬件友好**: 支持高效 INT4 推理

### 参考资料
- [AWQ Paper](https://arxiv.org/abs/2306.00978)
- [AWQ GitHub](https://github.com/mit-han-lab/llm-awq)